## Data Retrieval

In [16]:
import os
import glob
import pandas as pd

In [17]:
def _csv_instrument_key(path):
	basename = os.path.splitext(os.path.basename(path))[0]
	if "_" in basename:
		return basename.split("_", 1)[0]
	return basename


def load_csvs_from_folder(folder_path="data"):
	"""Recursively load all CSV files under `folder_path` into a dict of DataFrames.

	Keys are the instrument names derived from the CSV filenames.
	For example, `CRUDEOILM_historical_data.csv` becomes `CRUDEOILM`.

	Returns:
		dict: {instrument_name: pandas.DataFrame}
	"""
	folder = os.path.abspath(folder_path)
	if not os.path.isdir(folder):
		raise FileNotFoundError(f"Folder not found: {folder}")

	# Find CSVs recursively
	csv_paths = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
	dataframes = {}
	for path in csv_paths:
		key = _csv_instrument_key(path)
		try:
			# Detect if a 'date' column exists (case-insensitive) and parse it
			header_cols = pd.read_csv(path, nrows=0).columns.tolist()
			lower_cols = [c.lower() for c in header_cols]
			parse_dates = [header_cols[i] for i, c in enumerate(lower_cols) if c == 'date'] or None
			if parse_dates:
				df = pd.read_csv(path, parse_dates=parse_dates)
			else:
				df = pd.read_csv(path)
			dataframes[key] = df
		except Exception as e:
			print(f"Failed to load {path}: {e}")
	return dataframes


def load_req_csvs(folder_path="data", instruments=None):
	"""Load only CSVs whose file name matches one of the provided instruments.

	Args:
		folder_path (str): root folder to search (recursively).
		instruments (list[str]): list of instrument names to filter by. Matching is
			case-insensitive and checks if an instrument string appears anywhere in
			the CSV filename, such as `CRUDEOILM_historical_data.csv`.

	Returns:
		dict: {instrument_name: pandas.DataFrame} for matching files.
	"""
	if not instruments:
		raise ValueError("`instruments` must be a non-empty list of instrument names")

	# normalize instruments for case-insensitive matching
	inst_upper = [str(i).upper() for i in instruments]

	folder = os.path.abspath(folder_path)
	if not os.path.isdir(folder):
		raise FileNotFoundError(f"Folder not found: {folder}")

	csv_paths = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
	dataframes = {}
	for path in csv_paths:
		key = _csv_instrument_key(path)
		key_upper = key.upper()

		# include file if any instrument name is found in the CSV filename
		if not any(inst in key_upper for inst in inst_upper):
			continue

		try:
			header_cols = pd.read_csv(path, nrows=0).columns.tolist()
			lower_cols = [c.lower() for c in header_cols]
			parse_dates = [header_cols[i] for i, c in enumerate(lower_cols) if c == 'date'] or None
			if parse_dates:
				df = pd.read_csv(path, parse_dates=parse_dates)
			else:
				df = pd.read_csv(path)
			dataframes[key] = df
		except Exception as e:
			print(f"Failed to load {path}: {e}")

	return dataframes


def filter_close_price(df):
	"""Filter the DataFrame to include only 'date' and 'close' columns, if they exist."""
	cols = df.columns.str.lower()
	date_col = df.columns[cols == 'date'][0] if 'date' in cols else None
	close_col = df.columns[cols == 'close'][0] if 'close' in cols else None

	if date_col and close_col:
		return df[[date_col, close_col]].rename(columns={date_col: 'date', close_col: 'close'})
	else:
		print("Warning: 'date' or 'close' column not found. Returning original DataFrame.")
		return df

In [18]:
dfs = load_req_csvs("data", instruments=['NIFTY', 'BANKNIFTY', 'CRUDEOILM', 'GOLDGUINEA', 'NATGASMINI', 'SILVERMIC', 'USDINR', 'EURINR'])
print(f"Loaded {len(dfs)} CSVs: {list(dfs.keys())}")


for key in dfs:
    dfs[key] = filter_close_price(dfs[key])

dfs['NIFTY'].head()

Loaded 11 CSVs: ['EURINR', 'USDINR', 'CRUDEOILM', 'GOLDGUINEA', 'NATGASMINI', 'SILVERMIC', 'BANKNIFTY', 'FINNIFTY', 'MIDCPNIFTY', 'NIFTYNXT50', 'NIFTY']


,date,close
0,2021-05-28 00:00:00+05:30,15463.85
1,2021-05-31 00:00:00+05:30,15583.20
2,2021-06-01 00:00:00+05:30,15618.15
3,2021-06-02 00:00:00+05:30,15616.75
4,2021-06-03 00:00:00+05:30,15712.40


## Backadjustment

In [19]:
import calendar

Below function backadjusts data keeping rollover as a fixed date (Eg: 17th of every month). This is useful for MCX market contracts

In [20]:
def backadjust_expiry_date(df, rollover_day, date_col="date", price_col="close", adj_col="close_adj"):
    """
    Add additive back-adjustment on the given rollover day.
    If the rollover day is a holiday, the previous available trading date is used.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.tz_localize(None).dt.normalize()
    df = df.sort_values(date_col).reset_index(drop=True)
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

    if df.empty:
        return df

    available_dates = set(df[date_col])
    df["adj_offset"] = 0.0

    months = sorted({(d.year, d.month) for d in df[date_col]})
    for year, month in months:
        target_day = min(rollover_day, calendar.monthrange(year, month)[1])
        target = pd.Timestamp(year, month, target_day)

        if target > df[date_col].max():
            continue

        if target in available_dates:
            rollover_date = target
        else:
            prior = df[df[date_col] < target]
            if prior.empty:
                continue
            rollover_date = prior[date_col].max()

        idx = df.index[df[date_col] == rollover_date]
        if len(idx) == 0:
            continue
        idx = idx[0]
        if idx == 0:
            continue

        gap = df.at[idx, price_col] - df.at[idx - 1, price_col]
        df.loc[: idx - 1, "adj_offset"] += gap

    df[adj_col] = df[price_col] - df["adj_offset"]
    return df.drop(columns=["adj_offset"])

This function backadjusts data keeping the last given weekday as the expiry day (Eg: Last thursday of every month). This is useful for NFO contracts.

In [21]:
def _normalize_datetime_series(series):
    dt = pd.to_datetime(series, errors="coerce")
    if getattr(dt.dt, "tz", None) is not None:
        try:
            dt = dt.dt.tz_convert(None)
        except TypeError:
            dt = dt.dt.tz_localize(None)
    return dt.dt.tz_localize(None).dt.normalize()



In [22]:
def backadjust_expiry_fixed_day(df, expiry_day, date_col="date", price_col="close", adj_col="close_adj"):
    """Additive back-adjustment using a monthly expiry rule like 'last thursday'."""
    df = df.copy()
    df[date_col] = _normalize_datetime_series(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

    if df.empty:
        return df

    nth, weekday_idx = _parse_expiry_day(expiry_day)
    available_dates = set(df[date_col])
    df["adj_offset"] = 0.0

    months = sorted({(d.year, d.month) for d in df[date_col]})
    for year, month in months:
        expiry_date = _month_expiry_date(year, month, weekday_idx, nth)
        expiry_date = pd.Timestamp(expiry_date)
        if expiry_date.tzinfo is not None:
            expiry_date = expiry_date.tz_convert(None)
        expiry_date = expiry_date.normalize()
        if expiry_date > df[date_col].max():
            continue

        if expiry_date in available_dates:
            rollover_date = expiry_date
        else:
            prior = df[df[date_col] < expiry_date]
            if prior.empty:
                continue
            rollover_date = prior[date_col].max()

        idx = df.index[df[date_col] == rollover_date]
        if len(idx) == 0 or idx[0] == 0:
            continue

        idx = idx[0]
        gap = df.at[idx, price_col] - df.at[idx - 1, price_col]
        df.loc[: idx - 1, "adj_offset"] += gap

    df[adj_col] = df[price_col] - df["adj_offset"]
    return df.drop(columns=["adj_offset"])


This function keeps the expiry day as two days before the last trading day of every month. This is useful for CDS contracts.

In [23]:
def backadjust_two_days_before_last(df, date_col="date", price_col="close", adj_col="close_adj"):
    """Additive back-adjustment using the second-to-last trading day of each month."""
    df = df.copy()
    df[date_col] = _normalize_datetime_series(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

    if df.empty:
        return df

    df["adj_offset"] = 0.0
    grouped = df.groupby([df[date_col].dt.year, df[date_col].dt.month])

    for (year, month), group in grouped:
        if len(group) < 2:
            continue

        second_last_date = group[date_col].nlargest(2).iloc[-1]
        idx = df.index[df[date_col] == second_last_date]
        if len(idx) == 0 or idx[0] == 0:
            continue

        idx = idx[0]
        gap = df.at[idx, price_col] - df.at[idx - 1, price_col]
        df.loc[: idx - 1, "adj_offset"] += gap

    df[adj_col] = df[price_col] - df["adj_offset"]
    return df.drop(columns=["adj_offset"])


In [32]:
nifty_backadj = backadjust_expiry_fixed_day(dfs['NIFTY'], expiry_day="last thursday")

nifty_backadj.head()

,date,close,close_adj
0,2021-05-27,15463.85,10846.50
1,2021-05-30,15583.20,10965.85
2,2021-05-31,15618.15,11000.80
3,2021-06-01,15616.75,10999.40
4,2021-06-02,15712.40,11095.05


In [29]:
crudeoil_backadj = backadjust_expiry_date(dfs['CRUDEOILM'], rollover_day=17)

crudeoil_backadj.head()

,date,close,close_adj
0,2023-03-03,6531,6589.0
1,2023-03-06,6588,6646.0
2,2023-03-07,6449,6507.0
3,2023-03-08,6342,6400.0
4,2023-03-09,6321,6379.0


In [30]:
usdinr_backadj = backadjust_two_days_before_last(dfs['USDINR'])
usdinr_backadj.head()

,date,close,close_adj
0,2021-05-27,72.8150,67.9200
1,2021-05-30,72.8950,68.0800
2,2021-05-31,73.1650,68.3500
3,2021-06-01,73.3925,68.5775
4,2021-06-02,73.1175,68.3025
